In [ ]:
import json
import pandas as pd
import requests

# -----------------------
# STEP 1: Load JSON from remote URL
# -----------------------

json_url = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json"

response = requests.get(json_url)
response.raise_for_status()  # Raise an error for bad status codes

content = response.text

try:
    data = json.loads(content)
    print("✅ Loaded as standard JSON array.")
except json.JSONDecodeError:
    data = [json.loads(line) for line in content.splitlines() if line.strip()]
    print("✅ Loaded as NDJSON.")

df = pd.DataFrame(data)

# -----------------------
# STEP 1B: Prepare fields
# -----------------------
for field in ["title", "abstract", "publication_number"]:
    if field not in df.columns:
        df[field] = ""

df["combined_text"] = df["title"].fillna("") + " " + df["abstract"].fillna("")

# -----------------------
# STEP 2: Classify Domains & Subdomains
# -----------------------

subdomains = {
    "Traditional": {
        "Compressors": ["compressor", "compression", "pressurize", "scroll"],
        "Refrigerants": ["refrigerant", "working fluid", "coolant", "gas", "phase change"],
        "Heat Exchange": ["evaporator", "condenser", "heat exchange", "heat exchanger"],
        "Flexible Cycles": ["flexible cycle", "variable cycle", "multi-mode"],
        "Defrosting": ["defrost", "anti-frost", "ice removal"],
        "Thermal Storage": ["thermal storage", "phase change material", "PCM", "latent heat"],
        "Other": []
    },
    "System Design": {
        "Configuration": ["system design", "setup", "configuration"],
        "Controls": ["control", "controller", "sensor", "automation", "feedback", "adaptive"],
        "Operating Framework": ["operating framework", "logic", "algorithm", "interface"],
        "Other": []
    },
    "Non-Traditional": {
        "Elastocaloric": ["elastocaloric"],
        "Electrocaloric": ["electrocaloric"],
        "Magnetocaloric": ["magnetocaloric"],
        "Ionocaloric": ["ionocaloric"],
        "Barocaloric": ["barocaloric"],
        "Thermoelectric": ["thermoelectric"],
        "Electrochemical": ["electrochemical", "membrane", "chemisorption"],
        "Thermoacoustic": ["thermoacoustic"],
        "Other": []
    },
    "Manufacturing": {
        "Modular Assembly": ["modular", "assembly", "unitised", "snap-fit"],
        "Supply Chain / Materials": ["supply chain", "material", "sourcing"],
        "Disassembly / Remanufacturing": ["disassembly", "recyclable", "remanufacturing"],
        "Process Optimization": ["manufacture", "fabrication", "process", "automation"],
        "Other": []
    }
}

def classify_innovation(text):
    if not isinstance(text, str):
        return {}
    text_lower = text.lower()
    results = {domain: [] for domain in subdomains}
    results["Other"] = True
    for domain, subcats in subdomains.items():
        for subcat, keywords in subcats.items():
            if any(kw in text_lower for kw in keywords):
                results[domain].append(subcat)
                results["Other"] = False
    return results

# Apply classification
classifications = df["combined_text"].apply(classify_innovation)
classified_df = pd.json_normalize(classifications)

# Merge and export
output_df = pd.concat([df[["publication_number", "title", "abstract"]], classified_df], axis=1)
output_df.to_csv("step1_2_classified_patents.csv", index=False)

print("✅ Steps 1 & 2 complete. Output saved to 'step1_2_classified_patents.csv'")


In [1]:
import openai
from openai import OpenAI
client = OpenAI()

In [2]:
response = client.responses.create(
    model="gpt-4.1",
    instructions="Talk like a pirate.",
    input="Are semicolons optional in JavaScript?",
)

print(response.output_text)

Arrr, matey! When it comes to JavaScript, the use o’ semicolons be a topic hotter than a chest o’ cursed gold! In truth, semicolons **be not strictly required** most o’ the time, thanks to a thing called **Automatic Semicolon Insertion (ASI)**. The JavaScript engine oft tosses in them semicolons when ye forget ‘em.

But beware, ye scallywag! Though the language be forgiving, there be **pirate traps** if ye skip yer semicolons. Sometimes, ASI don’t work as ye expect, and yer code can break or do strange things.

**Best advice o’ the seas:** 
- Use semicolons, or follow a strict code style (like StandardJS) if ye dare sail without ‘em.
- Remember, not usin’ ‘em can lead to buggy behavior, matey.

So, no—semicolons ain’t always required, but usin’ ‘em might save yer code from walkin’ the plank! 🏴‍☠️


In [3]:
import pandas as pd
import json
from time import sleep
from dotenv import load_dotenv
import os
from openai import OpenAI
import re

# -------------------------------
# Load environment and OpenAI client
# -------------------------------
load_dotenv(dotenv_path="/home/pascualdiego/projects/DiscoveryHP/.env")  # adjust if needed
client = OpenAI()

# -------------------------------
# Configuration
# -------------------------------
INPUT_FILE = "step1_2_classified_patents.csv"
OUTPUT_FILE = "step3_llm_subdomain_output.csv"
MAX_RECORDS = 20  # limit for initial test

# -------------------------------
# Load classified patent data
# -------------------------------
df = pd.read_csv(INPUT_FILE).head(MAX_RECORDS).copy()

# -------------------------------
# Prompt builder
# -------------------------------
def make_prompt(title, abstract):
    return f"""
You are an expert in sustainable heating technologies.

Patent title: {title}
Abstract: {abstract}

Based on the title and abstract, classify this patent into all applicable subdomains across the following domains:

1. Traditional Components:
   - Compressors
   - Refrigerants
   - Heat Exchange
   - Flexible Cycles
   - Defrosting
   - Thermal Storage
   - Other

2. System Design:
   - Configuration
   - Controls
   - Operating Framework
   - Other

3. Non-Traditional Technologies:
   - Elastocaloric
   - Electrocaloric
   - Magnetocaloric
   - Ionocaloric
   - Barocaloric
   - Thermoelectric
   - Electrochemical
   - Thermoacoustic
   - Other

4. Manufacturing & Assembly:
   - Modular Assembly
   - Supply Chain / Materials
   - Disassembly / Remanufacturing
   - Process Optimization
   - Other

Respond strictly in this format:
Traditional: [subdomain1, subdomain2, ...]
System Design: [subdomain1, ...]
Non-Traditional: [subdomain1, ...]
Manufacturing: [subdomain1, ...]
"""

# -------------------------------
# OpenAI classification call
# -------------------------------
def classify_subdomains(row):
    try:
        prompt = make_prompt(row['title'], row['abstract'])

        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )

        reply = response.choices[0].message.content
        return reply

    except Exception as e:
        print(f"Error for {row['publication_number']}: {e}")
        return f"ERROR: {e}"

# -------------------------------
# Subdomain parsing
# -------------------------------
def extract_flags(text, domain, options):
    if not isinstance(text, str):
        return {f"{domain}_{opt}": False for opt in options}
    pattern = rf"{domain}:\s*\[(.*?)\]"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    found = match.group(1) if match else ""
    flags = {}
    for opt in options:
        flags[f"{domain}_{opt}"] = opt in found
    return flags

# Define all known subdomain tags for parsing
subdomains = {
    "Traditional": ["Compressors", "Refrigerants", "Heat Exchange", "Flexible Cycles", "Defrosting", "Thermal Storage", "Other"],
    "System Design": ["Configuration", "Controls", "Operating Framework", "Other"],
    "Non-Traditional": ["Elastocaloric", "Electrocaloric", "Magnetocaloric", "Ionocaloric", "Barocaloric", "Thermoelectric", "Electrochemical", "Thermoacoustic", "Other"],
    "Manufacturing": ["Modular Assembly", "Supply Chain / Materials", "Disassembly / Remanufacturing", "Process Optimization", "Other"]
}

# -------------------------------
# Run classification loop
# -------------------------------
results = []
for _, row in df.iterrows():
    print(f"Processing: {row['publication_number']}")
    response = classify_subdomains(row)
    row_result = {
        "publication_number": row["publication_number"],
        "title": row["title"],
        "abstract": row["abstract"],
        "llm_response": response
    }
    for domain, options in subdomains.items():
        flags = extract_flags(response, domain, options)
        row_result.update(flags)
    results.append(row_result)
    sleep(1.5)  # to respect OpenAI rate limits

# -------------------------------
# Save results
# -------------------------------
result_df = pd.DataFrame(results)
result_df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Step 3 complete. Saved {len(result_df)} records to: {OUTPUT_FILE}")


Processing: WO-2024184245-A1
Processing: US-11994321-B2
Processing: WO-2022023086-A1
Processing: CN-211854451-U
Processing: WO-2024162444-A1
Processing: US-2022318458-A1
Processing: CN-107957157-B
Processing: CN-216308282-U
Processing: CN-211400375-U
Processing: CN-215951559-U
Processing: CN-107525355-B
Processing: CN-109340867-B
Processing: US-11306942-B2
Processing: KR-20240000964-A
Processing: CN-108635894-B
Processing: US-10753655-B2
Processing: CN-108093675-B
Processing: KR-20210106196-A
Processing: CN-215260626-U
Processing: CN-210085240-U

✅ Step 3 complete. Saved 20 records to: step3_llm_subdomain_output.csv


In [1]:
import pandas as pd
import re
from collections import Counter
import matplotlib.pyplot as plt

# -------------------------------
# STEP 4: SETUP
# -------------------------------
INPUT_FILE = "step3_llm_subdomain_output.csv"
CSV_OUTPUT = "step4_application_context_keywords.csv"
PLOT_OUTPUT = "step4_keyword_barplot.png"

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv(INPUT_FILE)

# -------------------------------
# STEP 4A: Classify Application Contexts
# -------------------------------
application_keywords = {
    "Residential": ["home", "residential", "domestic", "household"],
    "Commercial": ["commercial", "building", "office", "HVAC"],
    "Industrial": ["industrial", "plant", "factory", "process heating"],
    "Transportation": ["vehicle", "automotive", "EV", "bus", "truck"],
    "District Heating": ["district heating", "central heating", "community"],
    "Water Heating": ["water heater", "hot water", "boiler"],
    "Drying Processes": ["drying", "dehydration"],
    "Battery Thermal Management": ["battery", "thermal management"],
}

def classify_application_context(text):
    if not isinstance(text, str):
        return {ctx: False for ctx in application_keywords}
    text_lower = text.lower()
    flags = {}
    for context, keywords in application_keywords.items():
        flags[context] = any(kw in text_lower for kw in keywords)
    return flags

# Apply context classification
context_flags = df["llm_response"].apply(classify_application_context)
context_df = pd.DataFrame(context_flags.tolist())

# -------------------------------
# STEP 4B: Extract Emerging Keywords
# -------------------------------
def extract_keywords(text):
    if not isinstance(text, str):
        return []
    words = re.findall(r"\b\w+\b", text.lower())
    return [
        word for word in words 
        if len(word) > 4 and word not in {
            "based", "which", "system", "device", "using", "provide", "allows", "title", "abstract"
        }
    ]

all_keywords = Counter()
df["llm_response"].dropna().apply(lambda x: all_keywords.update(extract_keywords(x)))
top_keywords = all_keywords.most_common(20)

# -------------------------------
# STEP 4C: Save Bar Chart of Top Keywords
# -------------------------------
keywords, counts = zip(*top_keywords)
plt.figure(figsize=(10, 6))
plt.barh(keywords[::-1], counts[::-1], color='steelblue')
plt.xlabel("Frequency")
plt.title("Top Emerging Keywords in LLM Responses")
plt.tight_layout()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.savefig(PLOT_OUTPUT)
plt.close()

# -------------------------------
# STEP 4D: Save Combined Output
# -------------------------------
df_combined = pd.concat([df, context_df], axis=1)
df_combined.to_csv(CSV_OUTPUT, index=False)

# -------------------------------
# COMPLETE
# -------------------------------
print(f"✅ Step 4 complete.")
print(f"📄 CSV saved to: {CSV_OUTPUT}")
print(f"📊 Keyword bar chart saved to: {PLOT_OUTPUT}")


✅ Step 4 complete.
📄 CSV saved to: step4_application_context_keywords.csv
📊 Keyword bar chart saved to: step4_keyword_barplot.png


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# CONFIGURATION
# -------------------------------
INPUT_FILE = "step4_application_context_keywords.csv"
OUTPUT_PREFIX = "step5"

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv(INPUT_FILE)

# -------------------------------
# STEP 5A: DOMAIN FREQUENCY SUMMARY
# -------------------------------
# Define column groups
domain_groups = {
    "Traditional": [col for col in df.columns if col.startswith("Traditional_")],
    "System Design": [col for col in df.columns if col.startswith("System Design_")],
    "Non-Traditional": [col for col in df.columns if col.startswith("Non-Traditional_")],
    "Manufacturing": [col for col in df.columns if col.startswith("Manufacturing_")]
}

# Count True values per subdomain
subdomain_counts = {}
for domain, cols in domain_groups.items():
    counts = df[cols].sum().sort_values(ascending=False)
    subdomain_counts[domain] = counts

# -------------------------------
# STEP 5B: VISUALIZE SUBDOMAIN FREQUENCIES
# -------------------------------
for domain, counts in subdomain_counts.items():
    plt.figure(figsize=(8, 5))
    counts.plot(kind='barh', color='skyblue')
    plt.xlabel("Frequency")
    plt.title(f"{domain} Subdomain Frequency")
    plt.tight_layout()
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    chart_file = f"{OUTPUT_PREFIX}_{domain.lower().replace(' ', '_')}_subdomains.png"
    plt.savefig(chart_file)
    plt.close()
    print(f"📊 Saved: {chart_file}")

# -------------------------------
# STEP 5C: APPLICATION CONTEXT FREQUENCIES
# -------------------------------
context_columns = [
    "Residential", "Commercial", "Industrial", "Transportation",
    "District Heating", "Water Heating", "Drying Processes", "Battery Thermal Management"
]

context_counts = df[context_columns].sum().sort_values(ascending=False)

# Plot
plt.figure(figsize=(8, 5))
context_counts.plot(kind='barh', color='orange')
plt.xlabel("Frequency")
plt.title("Application Context Frequency")
plt.tight_layout()
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.savefig(f"{OUTPUT_PREFIX}_application_contexts.png")
plt.close()

print(f"📊 Saved: {OUTPUT_PREFIX}_application_contexts.png")

# -------------------------------
# COMPLETE
# -------------------------------
print("✅ Step 5 complete. Subdomain and context trend plots generated.")


📊 Saved: step5_traditional_subdomains.png
📊 Saved: step5_system_design_subdomains.png
📊 Saved: step5_non-traditional_subdomains.png
📊 Saved: step5_manufacturing_subdomains.png
📊 Saved: step5_application_contexts.png
✅ Step 5 complete. Subdomain and context trend plots generated.


In [4]:
import pandas as pd
import os
from openai import OpenAI
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import hdbscan
from dotenv import load_dotenv
from tqdm import tqdm

# -------------------------------
# SETUP
# -------------------------------
load_dotenv("/home/pascualdiego/projects/DiscoveryHP/.env")  # Adjust path if needed
client = OpenAI()
INPUT_FILE = "step4_application_context_keywords.csv"
OUTPUT_EMBED_FILE = "step6_clustered_topics.csv"
EMBED_MODEL = "text-embedding-ada-002"  # PUBLIC and accessible

# -------------------------------
# LOAD AND PREP TEXTS
# -------------------------------
df = pd.read_csv(INPUT_FILE)
texts = df["llm_response"].fillna("").tolist()

# -------------------------------
# EMBEDDING GENERATION
# -------------------------------
def get_embedding(text, model=EMBED_MODEL):
    try:
        response = client.embeddings.create(
            input=text,
            model=model
        )
        return response.data[0].embedding
    except Exception as e:
        print("Embedding error:", e)
        return [0.0] * 1536  # Dummy fallback for consistent shape

print("🔄 Generating embeddings...")
embeddings = [get_embedding(t) for t in tqdm(texts)]

# -------------------------------
# DIMENSIONALITY REDUCTION + CLUSTERING
# -------------------------------
print("📉 Reducing dimensions + clustering...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
reduced = reducer.fit_transform(embeddings)

clusterer = hdbscan.HDBSCAN(min_cluster_size=5, prediction_data=True)
clusters = clusterer.fit_predict(reduced)

# -------------------------------
# VISUALIZATION
# -------------------------------
plt.figure(figsize=(10, 6))
palette = sns.color_palette("hls", len(set(clusters)))
sns.scatterplot(
    x=reduced[:, 0], y=reduced[:, 1], hue=clusters, palette=palette, legend="full", s=60
)
plt.title("Step 6: LLM-Based Topic Clusters")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig("step6_umap_clusters.png")
plt.close()
print("📊 Cluster plot saved: step6_umap_clusters.png")

# -------------------------------
# SAVE FINAL DATA
# -------------------------------
df["embedding"] = embeddings
df["topic_cluster"] = clusters
df["umap_x"] = reduced[:, 0]
df["umap_y"] = reduced[:, 1]
df.to_csv(OUTPUT_EMBED_FILE, index=False)

print(f"✅ Step 6 complete. Saved clustered data to: {OUTPUT_EMBED_FILE}")


🔄 Generating embeddings...


  5%|▌         | 1/20 [00:00<00:10,  1.86it/s]

Embedding error: Error code: 403 - {'error': {'message': 'Project `proj_IgUh1Le96lnS9TX7SgQF7Kse` does not have access to model `text-embedding-ada-002`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


 10%|█         | 2/20 [00:00<00:06,  2.93it/s]

Embedding error: Error code: 403 - {'error': {'message': 'Project `proj_IgUh1Le96lnS9TX7SgQF7Kse` does not have access to model `text-embedding-ada-002`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


...

Embedding error: Error code: 403 - {'error': {'message': 'Project `proj_IgUh1Le96lnS9TX7SgQF7Kse` does not have access to model `text-embedding-ada-002`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


100%|██████████| 20/20 [00:06<00:00,  3.08it/s]

Embedding error: Error code: 403 - {'error': {'message': 'Project `proj_IgUh1Le96lnS9TX7SgQF7Kse` does not have access to model `text-embedding-ada-002`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
📉 Reducing dimensions + clustering...



/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


📊 Cluster plot saved: step6_umap_clusters.png
✅ Step 6 complete. Saved clustered data to: step6_clustered_topics.csv
